# EXP02 — Sensibilidade ao filtro de transações

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Enviar e conferir os dados

O EXP01 e o EXP02 usam os mesmos arquivos Parquet processados da pesquisa. Baixe os arquivos a partir de um acesso autorizado à pasta oficial (veja `docs/DADOS.md`). A célula abaixo aceita os arquivos no Colab ou verifica os que já estiverem em `data/processed/` na execução local.

A verificação SHA-256 evita misturar versões diferentes. Ela não comprova que o pré-processamento original esteja correto.


In [ ]:
from pathlib import Path
import hashlib

ARQUIVOS_E_HASHES = {
    "features_treino.parquet": "0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32",
    "features_teste_20201029.parquet": "6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51",
}

pasta_dados = RAIZ / "data" / "processed"
pasta_dados.mkdir(parents=True, exist_ok=True)
faltantes = [nome for nome in ARQUIVOS_E_HASHES if not (pasta_dados / nome).is_file()]

if faltantes:
    try:
        import google.colab  # disponível apenas no Google Colab
    except ImportError as exc:
        raise FileNotFoundError(
            "Arquivos ausentes em data/processed/: " + ", ".join(faltantes)
        ) from exc

    from google.colab import files
    print("Envie os arquivos necessários:", ", ".join(faltantes))
    recebidos = files.upload()
    for nome, conteudo in recebidos.items():
        if nome not in ARQUIVOS_E_HASHES:
            raise ValueError("Arquivo não esperado: " + nome)
        (pasta_dados / nome).write_bytes(conteudo)

for nome, esperado in ARQUIVOS_E_HASHES.items():
    arquivo = pasta_dados / nome
    if not arquivo.is_file():
        raise FileNotFoundError("Arquivo ausente: " + str(arquivo))
    sha = hashlib.sha256()
    with arquivo.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            sha.update(bloco)
    encontrado = sha.hexdigest()
    if encontrado != esperado:
        raise ValueError(
            f"SHA-256 diferente em {nome}. Esperado: {esperado}; encontrado: {encontrado}"
        )
    print(f"Arquivo conferido: {nome} ({arquivo.stat().st_size} bytes)")


## 2. Dados e critério

Usamos o mesmo treino e o mesmo dia de 2020 do EXP01. Mudaremos **somente o mínimo de transações**, comparando 1, 5, 10 e 20. O limiar permanece `score > 0,50`.

In [ ]:
import pandas as pd
from ethmix.features import FEATURES
from ethmix.execucao import abrir_dados, treinar, prever

pasta = RAIZ / "data" / "processed"
treino = abrir_dados(pasta / "features_treino.parquet", ["label", "n_tx", *FEATURES])
teste = abrir_dados(pasta / "features_teste_20201029.parquet", ["recipient", "n_tx", "tornado", *FEATURES])
modelo, _ = treinar(treino, seed=42, min_tx=10)
registros = []
for minimo in [1, 5, 10, 20]:
    avaliados = prever(modelo, teste, min_tx=minimo, limiar=0.50)
    registros.append({"min_tx": minimo, "elegiveis": len(avaliados),
                      "alertas": int(avaliados["alerta"].sum())})
resultado = pd.DataFrame(registros)
display(resultado)

## 3. Visualizar

As contagens mostram o quanto o filtro altera a população avaliada. Menos alertas **não** significa, por si só, maior precisão.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(resultado["min_tx"].astype(str), resultado["alertas"])
ax.set_xlabel("Mínimo de transações")
ax.set_ylabel("Quantidade de alertas")
plt.show()
saida = RAIZ / "results" / "generated" / "EXP02"
saida.mkdir(parents=True, exist_ok=True)
resultado.to_csv(saida / "sensibilidade.csv", index=False)

## 4. Referência histórica

O relatório anterior registra **49, 14, 9 e 7 alertas** para mínimos de 1, 5, 10 e 20. A execução atual só confirma esses números se os arquivos, os parâmetros e o modelo forem equivalentes. O fato de o resultado de 1 transação permanecer abaixo de 63 mostra que o filtro, sozinho, não resolve a divergência.